### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import ast
import pickle
import os
import shutil
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

### 2. Load Datasets

In [2]:
try:
    books_df = pd.read_csv('data/books.csv', index_col=0)
    games_df = pd.read_csv('data/games.csv', index_col=0)
    movies_df = pd.read_csv('data/movies.csv', index_col=0)
    shows_df = pd.read_csv('data/shows.csv', index_col=0)
    print("All datasets loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading datasets: {e}. Please ensure all CSV files are in the correct directory.")

All datasets loaded successfully.


### 3. Clean and Unify Data

In [3]:
def clean_and_unify_data(books_df, games_df, movies_df, shows_df):
    # Add media_type
    books_df['media_type'] = 'Book'
    games_df['media_type'] = 'Game'
    movies_df['media_type'] = 'Movie'
    shows_df['media_type'] = 'Show'

    # Combine into a master dataframe
    master_df = pd.concat([books_df, games_df, movies_df, shows_df], ignore_index=True)

    # Create name_with_type
    master_df['name_with_type'] = master_df['name'] + ' (' + master_df['media_type'] + ')'

    # Clean genres
    def parse_genres(genres):
        if isinstance(genres, str):
            if genres.startswith('[') and genres.endswith(']'):
                try:
                    return ' '.join([i.lower() for i in ast.literal_eval(genres)])
                except (ValueError, SyntaxError):
                    return ''
            else:
                return genres.lower().replace(' & ', ' ').replace(',', ' ')
        return ''
    master_df['genres'] = master_df['genres'].apply(parse_genres)

    # Create tags
    master_df['tags'] = master_df['overview'].fillna('') + ' ' + master_df['genres'].fillna('') + ' ' + master_df['name'].fillna('')

    return master_df[['name_with_type', 'tags']]

master_df = clean_and_unify_data(books_df, games_df, movies_df, shows_df)
print('Data cleaning and unification complete.')
master_df.head()

Data cleaning and unification complete.


,name_with_type,tags
0,Drowned Wednesday (Book),Drowned Wednesday is the first Trustee among ...
1,The Lost Hero (Book),"As the book opens, Jason awakens on a school ..."
2,Magic's Promise (Book),The book opens with Herald-Mage Vanyel return...
3,The Sweet Far Thing (Book),The prologue begins with two men who are sear...
4,Master Alvin (Book),This book has yet to be published. At this ti...


### 4. Save Master Dataframe

In [4]:
pickle.dump(master_df, open('master_df.pkl', 'wb'))
print('Master dataframe saved as master_df.pkl')

Master dataframe saved as master_df.pkl


### 5. TF-IDF Vectorization

In [5]:
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
tfidf_matrix = tfidf.fit_transform(master_df['tags'])
print('TF-IDF vectorization complete.')
print(f'TF-IDF matrix shape: {tfidf_matrix.shape}')

TF-IDF vectorization complete.
TF-IDF matrix shape: (6658, 5000)


### 6. Compute and Save Cosine Similarity Matrix in Chunks

In [6]:
CHUNK_SIZE = 5000
SIMILARITY_DIR = 'similarity_chunks'

# Remove old similarity chunks if they exist
if os.path.exists(SIMILARITY_DIR):
    shutil.rmtree(SIMILARITY_DIR)
    print(f'Removed old directory: {SIMILARITY_DIR}')
os.makedirs(SIMILARITY_DIR)
print(f'Created new directory: {SIMILARITY_DIR}')

num_chunks = (tfidf_matrix.shape[0] + CHUNK_SIZE - 1) // CHUNK_SIZE

for i in range(num_chunks):
    start_row = i * CHUNK_SIZE
    end_row = min((i + 1) * CHUNK_SIZE, tfidf_matrix.shape[0])
    chunk = tfidf_matrix[start_row:end_row]
    sim_chunk = cosine_similarity(chunk, tfidf_matrix)
    chunk_file = os.path.join(SIMILARITY_DIR, f'sim_chunk_{start_row}.pkl')
    with open(chunk_file, 'wb') as f:
        pickle.dump(sim_chunk, f)
    print(f'Saved chunk {i+1}/{num_chunks} to {chunk_file}')

print('All similarity chunks have been computed and saved.')

Removed old directory: similarity_chunks
Created new directory: similarity_chunks
Saved chunk 1/2 to similarity_chunks/sim_chunk_0.pkl
Saved chunk 2/2 to similarity_chunks/sim_chunk_5000.pkl
All similarity chunks have been computed and saved.
